In [1]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "LiquidAI/LFM2.5-1.2B-Instruct",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


In [2]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 64,
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


```
<|startoftext|><|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

In [3]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Hello!"},
    {"role" : "assistant", "content" : "Hey there!"}
], tokenize = False)

'<|startoftext|><|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHey there!<|im_end|>\n'

In [6]:
from datasets import load_dataset
dataset = load_dataset("sarvamai/samvaad-hi-v1",split=["train"])

In [14]:
dataset=dataset[0]

In [15]:
dataset[0]

{'messages': [{'content': 'Can you tell me about the 1988 Rajya Sabha elections in English please?',
   'role': 'user'},
  {'content': "The 1988 Rajya Sabha elections were a significant event in Indian politics. The Rajya Sabha, which is the upper house of India's Parliament, sees periodic elections with members serving terms of six years. In 1988, elections took place in several states across India, electing new members to represent their respective constituencies.\n\nThe elected members were expected to serve from 1988 to 1994. However, there was a possibility for their term to end sooner due to circumstances such as resignation or death.\n\nAlso, apart from the main elections, specific special elections known as 'bye elections' also occurred in 1988. These are typically held to fill vacancies that arise during the tenure of the elected members.",
   'role': 'assistant'},
  {'content': 'Was it a norm for members to serve for a term of six years back then?',
   'role': 'user'},
  {'co

In [18]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        if messages:  # Skip empty messages
            text = tokenizer.apply_chat_template(
                messages,
                tokenize = False,
                add_generation_prompt = False,
            )
            texts.append(text.removeprefix(tokenizer.bos_token))
        else:
            texts.append("")  # Placeholder for empty messages
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/101476 [00:00<?, ? examples/s]

In [20]:
dataset[10]["text"]

'<|im_start|>user\nक्या भारतीय दर्शन और आध्यात्मिकता के बारे में कोई आम गलत धारणाएँ हैं जिन्हें आप स्पष्ट कर सकते हैं?<|im_end|>\n<|im_start|>assistant\nनिश्चित रूप से! भारतीय दर्शन और आध्यात्मिकता को अक्सर लोकप्रिय संस्कृति में गलत समझा जाता है या अति सरलीकरण किया जाता है। एक आम गलत धारणा यह है कि सभी भारतीय दर्शन और आध्यात्मिकता को "हिंदू धर्म" की छत्रछाया में समूहीकृत किया जा सकता है। जबकि हिंदू धर्म एक महत्वपूर्ण परंपरा है, जैन धर्म, बौद्ध धर्म, सिख धर्म और विभिन्न स्वदेशी परंपराओं सहित भारत में दर्शन और आध्यात्मिक प्रथाओं की विविधता को पहचानना महत्वपूर्ण है।\n\nएक और गलत धारणा यह है कि भारतीय आध्यात्मिकता पूरी तरह से पारगमन और दुनिया से अलगाव पर केंद्रित है। जबकि मोक्ष (मुक्ति) और त्याग जैसी अवधारणाएं वास्तव में मौजूद हैं, भारतीय दर्शन सभी जीवन के परस्पर जुड़ाव और समर्पण और अखंडता के साथ सामाजिक कर्तव्यों (धर्म) को पूरा करने के महत्व पर भी जोर देते हैं।\n\nइसके अलावा, इनाम और सजा की एक प्रणाली के रूप में कर्म की अवधारणा को अधिक सरल बनाने की प्रवृत्ति है। वास्तव में, कर्म की अवधारण

In [29]:
eval_size = min(1000, len(dataset))
train_size = len(dataset) - eval_size

train_dataset = dataset.select(range(train_size))
eval_dataset = dataset.select(range(train_size, len(dataset)))

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

Training samples: 100476
Evaluation samples: 1000


In [30]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,
        load_best_model_at_end = True,
        save_total_limit = 3,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "./checkpoints",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/100476 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [31]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=12):   0%|          | 0/100476 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [32]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|startoftext|><|im_start|>user\nCan you tell me about a longhorn beetle that is seen in India? Also, I want the information in Hindi.<|im_end|>\n<|im_start|>assistant\nभारत में पाए जाने वाले लॉन्गहॉर्न बीटल्स में से एक प्रकार ज्ञाथोले सिम्प्लेक्स है। यह बीटल कुछ विशेष पेड़ों की सॉफ्टवुड पर पालन करता है और इसका आयुस्मान एक वर्ष होता है। हालांकि, सूखे हालातों में यह तीन वर्ष तक जीने की क्षमता रखता है। यह मुख्य रूप से मई से अगस्त के बीच में देखा जाता है। साथ ही, इसके लार्वा विभिन्न पेड़ों जैसे कि हार्डविकिया बिनाटा, अलबिजिया ओडोरातिस्सिमा, मिलेटिया पिनाटा, पोंगामिया ग्लाब्रा, और शोरिया रोबुस्टा पर पालन करते हैं। भारत के अलावा, यह श्रीलंका, ताइवान, और म्यानमार जैसे देशों में भी पाया जाता है।<|im_end|>\n<|im_start|>user\nThis is so interesting! Can you tell me more about the plants it feeds on? And please reply in Hindi.<|im_end|>\n<|im_start|>assistant\nजरूर। हार्डविकिया बिनाटा, जिसे इंडियन बीच के नाम से भी जाना जाता है, यह भारतीय उपमहाद्वीप का मूल स्थानीय वृक्ष है और इसे उसकी अत्यंत टिक

In [33]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                  भारत में पाए जाने वाले लॉन्गहॉर्न बीटल्स में से एक प्रकार ज्ञाथोले सिम्प्लेक्स है। यह बीटल कुछ विशेष पेड़ों की सॉफ्टवुड पर पालन करता है और इसका आयुस्मान एक वर्ष होता है। हालांकि, सूखे हालातों में यह तीन वर्ष तक जीने की क्षमता रखता है। यह मुख्य रूप से मई से अगस्त के बीच में देखा जाता है। साथ ही, इसके लार्वा विभिन्न पेड़ों जैसे कि हार्डविकिया बिनाटा, अलबिजिया ओडोरातिस्सिमा, मिलेटिया पिनाटा, पोंगामिया ग्लाब्रा, और शोरिया रोबुस्टा पर पालन करते हैं। भारत के अलावा, यह श्रीलंका, ताइवान, और म्यानमार जैसे देशों में भी पाया जाता है।<|im_end|>\n                               जरूर। हार्डविकिया बिनाटा, जिसे इंडियन बीच के नाम से भी जाना जाता है, यह भारतीय उपमहाद्वीप का मूल स्थानीय वृक्ष है और इसे उसकी अत्यंत टिकाऊ लकड़ी के लिए जाना जाता है। अलबिजिया ओडोरातिस्सिमा भारत का एक और वृक्ष है जिसे ब्लैक सिरिस या काला सिरिस कहा जाता है, इसका विस्तृत गोलाकार कैनोपी और काली, घनी और टिकाऊ लकड़ी होती है। मिलेटिया पिनाटा, जिसे आमतौर पर भारतीय बीच या पोंगाम वृक्ष कहा जाता है, इस

In [34]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
5.738 GB of memory reserved.


In [35]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,476 | Num Epochs = 1 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,569,088 of 1,206,909,696 (3.03% trained)


Step,Training Loss,Validation Loss
20,1.052700,1.026857
40,1.004500,1.002960
60,0.836500,0.989599
80,0.970400,0.984238


In [36]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

856.2818 seconds used for training.
14.27 minutes used for training.
Peak reserved memory = 6.102 GB.
Peak reserved memory for training = 0.364 GB.
Peak reserved memory % of max memory = 27.694 %.
Peak reserved memory for training % of max memory = 1.652 %.


In [38]:
messages = [{
    "role": "user",
    "content": "अगर तुम्हें मौका मिले कि तुम एक ऐसा AI बनाओ जो भारत की किसी एक बड़ी समस्या को हल करे, तो तुम कौन-सी समस्या चुनोगे और क्यों?",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 512, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.3, min_p = 0.15, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

मौका 

मिलने पर, मैं आपको भारत की एक बड़ी समस्या को हल करने के लिए एक ऐसा AI बनाने का विकल्प देने का फैसला करूंगा। यह समस्या भारत की समाज के लिए एक महत्वपूर्ण समस्या है जो अपने समस्याओं को हल करने के लिए एक शक्तिशाली उपकरण की आवश्यकता रखता है।

इस समस्या को हल करने के लिए, मैं एक ऐसा AI बनाना चुनता हूं जो भारत की समाज की समस्याओं के लिए एक समग्र दृष्टिकोण प्रदान करता है। यह एक समस्या है जो भारत के समाज के लिए


In [39]:
messages = [{
    "role": "user",
    "content": "अगर तुम्हें अपनी ज़िंदगी में एक स्किल ऐसी चुननी हो जो आने वाले 10 सालों में सबसे ज़्यादा काम आए, तो वो कौन-सी होगी और क्यों?",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024, # Increase for longer outputs!
    # Recommended Liquid settings!
    temperature = 0.3, min_p = 0.15, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

जब 

तुम्हें अपनी ज़िंदगी में एक स्किल ऐसी चुननी हो जाती है जो आने वाले 10 सालों में सबसे ज़्यादा काम आएगी, तो यह एक स्किल होगी जो आपके जीवन के दौरान सबसे अधिक समय बिताएगी होगी। यह स्किल आपके जीवन के दौरान सबसे अधिक काम करने वाली होगी जो आपको सबसे अधिक समय बिताने वाली कार्यों को करने वाली होगी।

इसके अलावा, यह स्किल आपको सबसे अधिक प्रभावी कार्यों को करने वाली होगी जो आपको सबसे अधिक समय बिताने वाली होगी। यह स्किल आपको सबसे अधिक समय बिताने वाली कार्यों को करने वाली होगी, जैसे कि शुरुआती जीवन के दौरान कई कामों को करना और उन्हें समय बिताना।

इसके अलावा, यह स्किल आपको सबसे अधिक समय बिताने वाली कार्यों को करने वाली होगी जो आपको सबसे अधिक प्रभावी समय बिताने वाली होगी। यह स्किल आपको सबसे अधिक समय बिताने वाली कार्यों को करने वाली होगी, जैसे कि अपने जीवन के दौरान कई कामों को करना और उन्हें समय बिताना।

इसलिए, यह स्किल आ


In [40]:

model.push_to_hub_merged("wizardoftrap/LFM2.5-1.2B-hi-it", tokenizer,token = "hf_HlU*******************")

Found HuggingFace hub cache directory: /home/wizardoftrap_sp/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `wizardoftrap/LFM2.5-1.2B-hi-it`: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]


Successfully copied all 1 files from cache to `wizardoftrap/LFM2.5-1.2B-hi-it`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:36<00:00, 36.93s/it]


Unsloth: Merge process complete. Saved to `/home/wizardoftrap_sp/lfm-sft/wizardoftrap/LFM2.5-1.2B-hi-it`
